# BmD cube - Frontend orchestrator of the cubing engine

In previous tutorials we have put the emphasis on the software architecture that is defining for the cubing engine. In this tutorial we will discuss the orchestrator class `bmdCube` which serves as the entrypoint for users to request and process data as is required for them.

The `bmdCube` can be seen as an orchestrator of the data pipeline due to the fact that it takes the requested recipe and creates an instance of all dataset cubes such as `chelsaCube` or `gbifCube` and parses the recipe so that they fetch and process data as the user request. This allows the user to focus solely on understanding the recipe and the associated syntax to communicate what the end product of the cube should look like.

In this tutorial we will focus on the recipe structure and showcase how to utilize the produced cube in an efficient way.

Additionaly the recipe and code that has been used in this tutorial is provided as a template script in the config directory of the repository for immediate use

## Recipe structure & syntax

### General parameters

The first component of the recipe can be described as a general set of parameters that determine
- where the end product of the pipeline is written to
- what the directory name containing the cube will be called
- what the file format for the raster type data
- custom execution configuration


```yaml
base_dir: "./test_outputs/"
cube_name: "dev_cube"

# Only include this if you want to change the execution configuration 
# For more explanation of these options, consult the readme file in the config directory
"""
execution:
    min_slices_for_parallel: 16,
    max_safe_threadpool_workers: 8,
    large_slice_threshold_mb: 50.0,
    gdal_warp_block_size: 2048,
    warp_cache_cleanup_delay_sec: 30
"""

export_as:
  format: "zarr"
```

The combination of `base_dir` and `cube_name` will form the output directory to which all the results are written to. In addition to this we allow for custom execution parameters to be defined within the recipe. These optional parameters are commented out in the template as standard users should not change to much in these settings, experienced developers can tweak these with respect to the computational infrastructure that is available to them. 

### Spatial & temporal parameters

```yaml
# ======================================================================
# 1. SPATIAL & TEMPORAL SELECTION
# ======================================================================
spatial:
  target_grid: "EEA"
  target_resolution: "1km"
  use_bbox: true  
  bbox: #Flanders bbox
      long_min: 2.591733
      long_max: 5.883999
      lat_min: 50.680797
      lat_max: 51.963900
temporal:
  start_year: 2004  #start year of the european invasion of the asian hornet
  start_month: 1
  end_year: 2025
  end_month: 12
  time_aggregation: "monthly"
```

The spatial and temporal parameters define the extent to which the requested datasets are queried and processed. For the cubing engine we have access to different grids to which the data can be harmonized

```python
Available Master Grids:
 - EEA_10m: EPSG:3035 | Res: 10
 - EEA_100m: EPSG:3035 | Res: 100
 - EEA_250m: EPSG:3035 | Res: 250
 - EEA_500m: EPSG:3035 | Res: 500
 - EEA_1km: EPSG:3035 | Res: 1000
 - EEA_10km: EPSG:3035 | Res: 10000
 - Global_EqualArea_10m: EPSG:6933 | Res: 10
 - Global_EqualArea_100m: EPSG:6933 | Res: 100
 - Global_EqualArea_250m: EPSG:6933 | Res: 250
 - Global_EqualArea_500m: EPSG:6933 | Res: 500
 - Global_EqualArea_1km: EPSG:6933 | Res: 1000
 - Global_EqualArea_10km: EPSG:6933 | Res: 10000
 - Global_WGS84_0_3sec: EPSG:4326 | Res: 8.333333333333333e-05
 - Global_WGS84_3sec: EPSG:4326 | Res: 0.0008333333333333333
 - Global_WGS84_7_5sec: EPSG:4326 | Res: 0.0020833333333333333
 - Global_WGS84_15sec: EPSG:4326 | Res: 0.004166666666666667
 - Global_WGS84_30sec: EPSG:4326 | Res: 0.008333333333333333
 - Global_WGS84_5min: EPSG:4326 | Res: 0.08333333333333333
```

When constructing the recipe you provide either `EEA`, `Global_EqualArea` or `Global_WGS84` in the `target_grid` section. For the `resolution` section the options are `10m`, `100m`, `250m`, `500m`, `1km`, `10km`, `0_3sec`, `3sec`, `7_5sec`, `15sec`, `30sec` or `5min`.

For the temporal parameters we operate on closed intervals, i.e. data will be sampled within `[start_year, end_year]` and `[start_month, end_month]`

### Datasource section

#### Generic datasource

For simple datasources such as `CHELSA` the recipe serves as a sort of switchboard where the user can toggle which variables are to be included in the produced cube. If we use `chelsa` as an example 

```yaml
# --------------------------------------------------------------------
  # CHELSA DATASETS (High-Resolution Climatologies)
  # --------------------------------------------------------------------
  chelsa:
    enabled: true
    catalog_path:  ""
    levels: 
      # --- DAILY TIME SERIES (Inherits global 'temporal' block) ---
      daily:
        include: true
        variables:
          tas: true     # Mean daily air temperature
          tasmax: false # Maximum daily air temperature
          tasmin: false # Minimum daily air temperature
          pr: false     # Daily precipitation amount
          clt: false    # Cloud area fraction
          rsds: false   # Surface downwelling shortwave radiation
          sfcWind: false # Near-surface wind speed
          hurs: false   # Near-surface relative humidity
          vpd: false    # Vapor pressure deficit
```

We see that in order to include it we set the `enabled` boolean variable to true followed by the levels we wish to include (for CHELSA these levels are `daily`, `monthly`, `annual`, `bioclim`, `climatologies`). In the library an example of all options is provided within the `template_recipe.yaml` file located in the `config` folder in the base directory. Variables or levels that **are not of interest to the user do not need to be included explicitly in the recipe (these are shown here and in the template explicitly to highlight the reasoning of the recipe syntax**. A catalog is required for the engine to properly function, this is a file that follows the structure outlined in the `2a_rasterCube.ipynb` and `2b_vectorCube.ipynb` tutorials meant for backend developers. Users can provide their own catalog to the recipe however, within the dataset adaptor classes the development team has incorporated standard catalogs for each dataset stored in the `meta` directory in the base folder. As such, users do not need to be concerned with this specific option and can level it as an empty string 

#### GBIF

GBIF represents a more complex recipe syntax due to the fact that we offer interfacing with the GBIF SQL API required for downloading live data from their servers.

```yaml
gbif:
    #When the user wishes to load in a file on disc provide path here
    local_file_path: 
    # =======================================================================
    # 1. QUERY & FILTER CONFIGURATION
    # Defines exactly what data is pulled from the GBIF network.
    # =======================================================================
    query_filters:
      # General taxonKey parser; accepts keys from all taxonomic levels (e.g., family, genus, species).
      taxon_keys: ['G59D','4F6YZ','3Y9W2','5B9T3','4HPXM','3Y9VW','75SVV','3ZW55','3ZDRX','6CJDB','GGZL','6JYHZ','7G3C6']
      
      # If true, ignores taxon_keys and fetches ALL occurrence data within the Area of Interest (AOI).
      fetch_all_taxa: false
      
      # Determines the type of records to fetch. Options: "presence", "absence", or "mixed" (both).
      record_type: "presence"
      
      # Fallback coordinate uncertainty (in meters) assigned to records missing this value.
      # If left blank, defaults to matching the spatial target_resolution.
      default_Uncertainty: 1000
      
      # Hard cutoff for coordinate uncertainty. Records exceeding this radius (in meters) are dropped.
      max_uncertainty: 1000
      
      # GBIF data quality flags to exclude. Drops records flagged with these specific issues.
      exclude_issues:
        - "ZERO_COORDINATE"
        - "COORDINATE_OUT_OF_RANGE"
        - "COUNTRY_COORDINATE_MISMATCH"
    
    # =======================================================================
    # 2. TAXONOMIC HARMONIZATION
    # Controls how taxonomic lineages are resolved and appended.
    # =======================================================================
    taxonomy:
      # If true, cross-references and aligns taxa against the Catalogue of Life (CoL) backbone.
      col_backbone: true
      
      # Defines the deepest taxonomic rank to include (kingdom, phylum, class, order, family, genus).
      # If left empty, only the species key will be retained as an identifier in the final table.
      lowest_level: 

    # =======================================================================
    # 3. ATTRIBUTE RETENTION
    # =======================================================================
    # Additional Darwin Core columns to extract and retain alongside the geometries.
    columns: ["year", "month"]
    time_cols: ["year", "month"] #Information for the CF temporal conversion
```

Based on the entries that the user provides in the recipe the `bmc` library constructs a SQL query that can be submitted to the GBIF server to retrieve all required data. If the user wishes to skip the download using a GBIF SQL query they can alternatively provide a darwincore csv table stored at a specified location which can be fed to the cubing engine on the `local_file_path` field. Explanation for each field is provided in the template recipe as shown above.

Processing the GBIF data requires the specification of a `vector_processing` field that is similarly structured for different datasources

#### `vector_processing`

When handling vector data the user can make specific requests how the data should be processed and aggregated in the resulting cube

```yaml
# =======================================================================
    # 4. SPATIAL INGESTION & PROCESSING ROUTING
    # =======================================================================
    # Defines exactly how the pipeline handles spatial geometries.
    # Options: 
    #  - "raw": Bypasses all grid math. Returns raw, unmapped occurrence records.
    #  - "vector": Downloads raw points, applies geometric topologies (buffers/clouds), and maps to target grid.
    #  - "api_cube": Uses GBIF's server-side pre-aggregated SQL grid API (fast, contracted data).
    processing_mode: "vector"

    # -----------------------------------------------------------------------
    # ROUTE A: PURE VECTOR PROCESSING (Used when processing_mode: "vector")
    # Handles raw coordinate mathematics and spatial transformations.
    # -----------------------------------------------------------------------
    vector_processing:
      # Desired geometric topology: "point", "polygon", or "point_cloud".
      topology: "point_cloud"
      
      # Mapping output: "classification" (1-to-1 cell assignment) or 
      # "fractional" (1-to-N probability/areal weights).
      mapping_mode: "fractional"  
      
      # Spatial lookup strategy: "intersect" (precise overlap) or "kdtree" (nearest centroid).
      # Note: KDTree is only mathematically valid when mapping_mode is 'classification'.
      spatial_method: "intersect"

      # Topology-specific configurations passed directly to the geometry engines.
      # The orchestrator will only apply the config matching the active 'topology'.
      topology_config:
        point_cloud:
          # Number of randomized points generated in the point cloud
          n_passes: 100
          
          # Underlying probability distribution (either "gaussian" or "uniform") 
          # with a width determined by the uncertainty radius
          distribution: "gaussian"
          
          # Random seed generator to ensure reproducibility
          random_seed: 42

        polygon:
          # Number of quadrant segments of the polygon determining the smoothness of the polygon being constructed.
          # 2-4 is very coarse, 16-32 is very smooth, 8 is the default value in many libraries.
          # Note: Increasing this will increase the impact on RAM memory during intersection operations.
          quad_segs: 8

    # -----------------------------------------------------------------------
    # ROUTE B: GBIF API CUBE CONFIGURATION (Used when processing_mode: "api_cube")
    # Maps pre-aggregated GBIF SQL grid cells using cell collection routines.
    # -----------------------------------------------------------------------
    api_cube_config:
      # "discrete" (integer sums/majority rule) or "continuous" (areal weighting).
      data_type: "discrete"       
      
      # "intersect" (geometric overlap/clipping) or "kdtree" (centroid snapping).
      spatial_method: "intersect"

    # =======================================================================
    # 5. DATA AGGREGATION & METRICS
    # =======================================================================
    aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["year", "month"]
      
      metrics:
        # 1. Expected Occurrences (Sums the geometric fractions)
        - column: "areal_fraction"
          method: "sum"
          weighted: false
          rename: "expected_occurrences"
          
        # 2. Species Count / Richness
        - column: "speciesKey"
          method: "nunique"
          weighted: false
          rename: "species_richness"
          
        # 3. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "distinct_observers"
```

##### Aggregation

```yaml
    # =======================================================================
    # 5. DATA AGGREGATION & METRICS
    # =======================================================================
    aggregate:
      export_unaggregated: true
      
      # Defines the non-spatial dimensions of the final data cube.
      # The spatial cell ID (grid_idx) is automatically appended by the engine.
      group_by_columns: ["year", "month"]
      
      metrics:
        # 1. Expected Occurrences (Sums the geometric fractions)
        - column: "areal_fraction"
          method: "sum"
          weighted: false
          rename: "expected_occurrences"
          
        # 2. Species Count / Richness
        - column: "speciesKey"
          method: "nunique"
          weighted: false
          rename: "species_richness"
          
        # 3. Distinct Observers
        - column: "recordedBy"
          method: "nunique"
          weighted: false
          rename: "distinct_observers"
```

Because the `aggregate_vector_cube` engine passes the `method` parameter directly to pandas' `agg()` function, it supports any standard pandas aggregation string.

| Method | Explanation | Common Geospatial/Biodiversity Use Case |
| :--- | :--- | :--- |
| **`count`** | Counts the number of non-null rows in a group. | Total number of observation records or telemetry pings in a grid cell. |
| **`nunique`** | Counts the number of distinct, unique values. | **Species Richness** (unique species), distinct habitats, or unique observers. |
| **`sum`** | Adds all numeric values together. | Total population counts, total biomass, or total deforested area. |
| **`mean`** | Calculates the arithmetic average. | Average tree height, mean annual temperature, or average confidence score. |
| **`median`** | Finds the middle value (50th percentile). | Robust average for highly skewed data, like extreme precipitation events. |
| **`min`** | Returns the smallest value. | Minimum elevation, earliest observation date, or lowest temperature. |
| **`max`** | Returns the largest value. | Highest IUCN threat level present, maximum canopy height, or latest sighting. |
| **`std`** | Calculates the standard deviation. | Environmental heterogeneity (e.g., variance in elevation within a cell). |
| **`var`** | Calculates the variance. | Statistical distribution of species abundance counts. |
| **`first` / `last`** | Returns the first or last non-null value. | Extracting the primary habitat type if the data is sorted by dominance. |

---

**Understanding the `weighted` Field**

In spatial vector processing, geometries rarely align perfectly with your analysis grid. A single polygon (e.g., a national park or an animal's home range) will frequently shatter across multiple grid cells. The `weighted` parameter determines how your engine handles the quantitative data attached to those shattered pieces.

Setting `weighted: true` activates a specific mass-preservation logic:

1. **The Spatial Multiplier:** The engine looks for a column like `areal_fraction`. If a polygon was split, this fraction represents how much of the original polygon landed in the current cell.
2. **The Calculation:** It multiplies the raw data value by this fraction *before* performing the aggregation (`data['_base_weight'] * numeric_series`).

**The "Phantom Population" Problem**

Imagine a single forest polygon that contains a known population of **500 birds**. When you overlay your grid, this polygon splits:
* **70%** of the polygon falls in Grid Cell A (`areal_fraction` = 0.7)
* **30%** of the polygon falls in Grid Cell B (`areal_fraction` = 0.3)

If you use `method: "sum"` with **`weighted: false`**:
* Cell A gets 500 birds.
* Cell B gets 500 birds.
* **Total:** Your dataset now reports 1,000 birds. You have fabricated 500 phantom birds just by placing a grid over the map.

If you use `method: "sum"` with **`weighted: true`**:
* Cell A gets 500 * 0.7 = **350 birds**.
* Cell B gets 500 * 0.3 = **150 birds**.
* **Total:** 500 birds. The biological mass is conserved and accurately apportioned across the landscape.

**The Engine Constraint**

`weighted: true` flag will **only** alter the calculation if the method is `sum` or `mean`. If a user sets `weighted: true` alongside `method: "nunique"`, the engine safely ignores the weight and calculates standard unique counts. This is mathematically correct: a species is either present in a grid cell or it isn't; you cannot have a fraction of a unique species ID.

## Generating a test cube

### Example recipe

For the example recipe we will focus on the Sonian Forest in the Brussels Capitol region in Belgium. For this natura2000 site we will extract all available data between 2018-2020 i.e. all chelsa data, request all species observations and request all CLMS data hosted on the copernicus dataspace

In [3]:
from pygbif import occurrences

# 1. Define the bounding box (min, max)
# Format: 'min,max'
lat_range = '50.723880,50.847357'  # e.g., rough latitudes for Flanders
lon_range = '4.326938,4.561834'    # e.g., rough longitudes for Flanders

# 2. Define the time frame
# Format: 'YYYY-MM-DD,YYYY-MM-DD' (or simply 'YYYY,YYYY' for the 'year' parameter)
date_range = '2018-01-01,2020-12-31'

# 3. Query the GBIF API
# Setting limit=0 ensures you only get the count, saving time and bandwidth
result = occurrences.search(
    decimalLatitude=lat_range,
    decimalLongitude=lon_range,
    eventDate=date_range,
    hasCoordinate=True, # Optional: ensures records actually have spatial data
    limit=0
)

# 4. Extract the count
total_records = result['count']

print(f"Total records found: {total_records}")

Total records found: 146133


In [1]:
tutorial_cube = {
    "base_dir": "./bmdCube_tutorial_output/",
    "cube_name": "eeaGrid100m_cube",
    "export_as": {
        "format": "netcdf"
    },
    "spatial": {
        "target_grid": "EEA",
        "target_resolution": "100m",
        "use_bbox": True,
        "bbox": {
            "long_min": 4.326938,
            "long_max": 4.561834,
            "lat_min": 50.723880,
            "lat_max": 50.847357
        }
    },
    "temporal": {
        "start_year": 2018,
        "start_month": 1,
        "end_year": 2020,
        "end_month": 12,
        "time_aggregation": "monthly"
    },
    "sources": {
        "gbif": {
            "local_file_path": None,
            "query_filters": {
                "fetch_all_taxa": True,
                "record_type": "presence",
                "default_Uncertainty": 1000,
                "max_uncertainty": 5000,
                "exclude_issues": [
                    "ZERO_COORDINATE",
                    "COORDINATE_OUT_OF_RANGE",
                    "COUNTRY_COORDINATE_MISMATCH"
                ]
            },
            "taxonomy": {
                "lowest_level": None
            },
            "columns": ["year", "month", "recordedby"],
            "time_cols": ["year", "month"],
            "processing_mode": "vector",
            "vector_processing": {
                "topology": "point_cloud",
                "mapping_mode": "classify",
                "spatial_method": "intersect",
                "topology_config": {
                    "point_cloud": {
                        "n_passes": 200,
                        "distribution": "uniform",
                        "random_seed": 42
                    }
                }
            },
            "aggregate": {
                "export_unaggregated": True,
                "group_by_columns": ["year", "month", "speciesKey"],
                "metrics": [
                    {
                        "column": "speciesKey",
                        "method": "count",
                        "weighted": False,
                        "rename": "occurrences"
                    },
                    {
                        "column": "recordedBy",
                        "method": "nunique",
                        "weighted": False,
                        "rename": "distinct_observers"
                    }
                ]
            }
        },
        "chelsa": {
            "enabled": True,
            "catalog_path": "C:/Users/niels/Documents/Repositories/BmC/meta/chelsa_gp_stac/chelsa_master.parquet",
            "levels": {
                "monthly": {
                    "include": True,
                    "variables": {
                        "tas": True,
                        "tasmax": True,
                        "tasmin": True
                    }
                },
                "annual": {
                    "include": True,
                    "variables": {
                        "qkndvi": True,
                        "mymd": True
                    }
                },
                "climatologies": {
                    "include": True,
                    "time_ranges": {
                        "1981-2010": True,
                        "2011-2040": True,
                        "2041-2070": True
                    },
                    "ensembles": {
                        "historical": True,
                        "GFDL-ESM4": True,
                        "IPSL-CM6A-LR": True,
                        "MPI-ESM1-2-HR": True,
                        "MRI-ESM2-0": True,
                        "UKESM1-0-LL": True
                    },
                    "scenarios": {
                        "historical": True,
                        "ssp126": True,
                        "ssp370": True,
                        "ssp585": True
                    },
                    "variables": {
                        "tas": True,
                        "tasmax": True,
                        "tasmin": True
                    }
                },
                "bioclim": {
                    "include": True,
                    "time_ranges": {
                        "1981-2010": True,
                        "2011-2040": True,
                        "2041-2070": True,
                        "2071-2100": True
                    },
                    "ensembles": {
                        "historical": True,
                        "GFDL-ESM4": True,
                        "IPSL-CM6A-LR": True,
                        "MPI-ESM1-2-HR": True,
                        "MRI-ESM2-0": True,
                        "UKESM1-0-LL": True
                    },
                    "scenarios": {
                        "historical": True,
                        "ssp126": True,
                        "ssp370": True,
                        "ssp585": True
                    },
                    "variables": {
                        "bio01": True,
                        "bio02": True,
                        "bio03": True,
                        "bio04": True,
                        "bio05": True,
                        "bio06": True,
                        "bio07": True,
                        "bio08": True,
                        "bio09": True,
                        "bio10": True,
                        "bio11": True,
                        "bio12": True,
                        "bio13": True,
                        "bio14": True,
                        "bio15": True,
                        "bio16": True,
                        "bio17": True,
                        "bio18": True,
                        "bio19": True,
                    }
                }
            }
        },
        "clms": {
                    "enabled": True,
                    "base_s3_path": "s3://eodata/CLMS/",
                    "levels": {
                        "clc": {
                            "include": True,
                            "variables": {
                                "landcover": True
                            }
                        },
                        "clcplus": {
                            "include": True,
                            "variables": {
                                "LULUCF-INSTANCE": True
                            }
                        },
                        "crop_types": {
                            "include": True,
                            "variables": {
                                "CTY": True
                            }
                        },
                        "dominant_leaf_type": {
                            "include": True,
                            "variables": {
                                "DLT": True
                            }
                        },
                        "forest_type": {
                            "include": True,
                            "variables": {
                                "FTY": True
                            }
                        },
                        "grassland_herbaceous": {
                            "include": True,
                            "variables": {
                                "GRA": True,
                                "HER": True
                            }
                        },
                        "tree_cover_density": {
                            "include": True,
                            "variables": {
                                "TCD": True
                            }
                        }
                    }
                }
            }
        }

The cubing engine makes calls to different API's and repositories to fetch all the data that is requested by the end user of the engine. In order to obtain access to all these different sources the user will need to provide credentials for each service that they wish to access. 

1) **CDSE**, The Copernicus Dataspace s3 data is available through the cubing engine but requires registration through their portal. More information about [how to request credentials](https://documentation.dataspace.copernicus.eu/APIs/S3.html) can be found in the provided link.
2) **GBIF**, if the user wishes to access provided by GBIF through their SQL API the user will need to make an account on the [GBIF portal](https://www.gbif.org/).
3) Currently, some CLMS data is hosted internally on the BMD dataspace s3 bucket. Access to this bucket is limited internally and credentials must be requested for this through the appropriate person. If a user is external to the BMD organization they can still utilize their own private bucket or host the data locally. Each datasource that involves data that is **not accessible through an API** will have builtin options for the user to point to locally stored data through a **{datasetName}_catalog_local.parquet** file stored in the `meta/` directory in the repository or through providing a custom catalog that points to s3 adresses combined with credentials for the private bucket.  

The CDSE credentials need to be stored in the following format (json)

```json
{
    "s3_endpoint" : "https://eodata.dataspace.copernicus.eu",
    "s3_access_key" : "yourAccessKey",
    "s3_secret_key" : "yourSecretKey"
}
```

The GBIF credentials need to be stored in the following format (json)

```json
{
    "GBIF_USER": "yourAccountName", 
    "GBIF_PWD": "yourPassword", 
    "GBIF_MAIL": "yourEmail"
}
```

Private credentials need to be stored in the following format (json)
```json
{
    "s3_access_key": "yourAccessKey", 
    "s3_secret_key": "yourSecretKey",
    "s3_endpoint": "https://s3.eu-north-1.amazonaws.com"
}
```

For the private credentials you need to explicitly mention which region the endpoint is stored in order to establish a proper connection

In [2]:
from bmc.cube.orchestrator.bmd import bmdCube
from pathlib import Path

cdse_creds = Path.home() / "cdse.json"
private_creds = Path.home() / "bmd.json"
gbif_creds = Path.home() / "gbif.json"

active_credentials = {
    "gbif_credentials": str(gbif_creds),
    "cdse_s3_credentials": str(cdse_creds),
    "private_s3_credentials": str(private_creds)
    }

orchestrator = bmdCube()
orchestrator.process_recipe(tutorial_cube, creds=active_credentials)

--- Generating Execution & Environment Provenance ---
--- Submitting GBIF Query (Running in Background) ---
Constructed and resolved master grid key: EEA_100m
Computing safe fetch envelope for target grid 'EEA_100m'...
source_resolution omitted for custom CRS 'EPSG:4326'. Applying default fallback: 0.008333333333333333
Safe Source Envelope (EPSG:4326): (4.27031, 50.67094, 4.61781, 50.90027)


INFO:Your sql download key is 0014085-261110092616502
[2026-11-18 09:12:37] [INFO] No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
INFO:No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
[2026-11-18 09:12:37] [INFO] 
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'chelsacube' ===
INFO:
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'chelsacube' ===
[2026-11-18 09:12:37] [INFO]   Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
INFO:  Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
[2026-11-18 09:12:37] 

GBIF download submitted successfully. Key: 0014085-261110092616502
Initializing CHELSA cube generation...


[2026-11-18 09:12:38] [INFO] Loaded catalog for level 'monthly' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
INFO:Loaded catalog for level 'monthly' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
[2026-11-18 09:12:39] [INFO] Loaded catalog for level 'annual' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
INFO:Loaded catalog for level 'annual' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
[2026-11-18 09:12:40] [INFO] Loaded catalog for level 'climatologies' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
INFO:Loaded catalog for level 'climatologies' (339225 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\chelsa\chelsa_catalog.parquet
[2026-11-18 09:12:40] [INFO] Loaded catalog for level 'bioclim' (339225 rows) from C:\Users\niels\Docu

Sanitizing spatial geometry...
Preparing out-of-core reprojection to EEA_1km...
Utilizing 'bilinear' resampling for reprojection.
Lazy xarray object detected. Preparing for disk stream...
Sanitizing spatial geometry...
Warping to ./bmdCube_tutorial_output/warp_cache\annual\mymd\slice_0000.tif (Resampling: bilinear)...
Reprojection complete.


[2026-11-18 09:13:15] [INFO] 
Processing Level: 'annual' | Variable: 'qkndvi'...
INFO:
Processing Level: 'annual' | Variable: 'qkndvi'...
[2026-11-18 09:13:15] [INFO] Initiating network fetch for 1 asset(s) using 1 worker(s) (Sub-linear sqrt(1) thread allocation)...
INFO:Initiating network fetch for 1 asset(s) using 1 worker(s) (Sub-linear sqrt(1) thread allocation)...
INFO:Initiating parallel raster fetch across 1 assets using 1 workers (Geometry: bounding box)...
INFO:Parallel ingest completed successfully with zero fetch failures.
[2026-11-18 09:13:16] [INFO] Processing data through chunked timeseries pipeline
INFO:Processing data through chunked timeseries pipeline


... (57 outputs left out)

INFO:On disk at ./bmdCube_tutorial_output/downloads/0014085-261110092616502.zip
[2026-11-18 09:51:42] [INFO] No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
INFO:No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
[2026-11-18 09:51:42] [INFO] 
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'gbif' ===
INFO:
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'gbif' ===
[2026-11-18 09:51:42] [INFO]   Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
INFO:  Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
[2026-11

Initializing GBIF cube generation...


[2026-11-18 09:51:43] [INFO] Aligning local column casing to match recipe: {'specieskey': 'speciesKey', 'recordedby': 'recordedBy'}
INFO:Aligning local column casing to match recipe: {'specieskey': 'speciesKey', 'recordedby': 'recordedBy'}
[2026-11-18 09:53:14] [INFO] Constructed and resolved master grid key: EEA_100m
INFO:Constructed and resolved master grid key: EEA_100m
[2026-11-18 09:53:14] [INFO] Extracting padded global bounding box from recipe...
INFO:Extracting padded global bounding box from recipe...
[2026-11-18 09:53:15] [INFO] CRS mismatch detected. Reprojecting from WGS 84 to ETRS89-extended / LAEA Europe...
INFO:CRS mismatch detected. Reprojecting from WGS 84 to ETRS89-extended / LAEA Europe...
[2026-11-18 10:01:07] [INFO] === Initiating Dynamic Vector QA/QC Profiling ===
INFO:=== Initiating Dynamic Vector QA/QC Profiling ===
[2026-11-18 10:01:12] [INFO] Validation report saved to ./bmdCube_tutorial_output/eeaGrid100m_cube\gbif\validation_report\gbif_qa_qc_failures.json
I

--- Constructing Unifying STAC Catalog (eeaGrid100m_cube_catalog) ---
Exported unifying STAC Catalog to: ./bmdCube_tutorial_output/eeaGrid100m_cube\meta\STAC\catalog.json
Successfully removed temporary STAC assets directory: ./bmdCube_tutorial_output/eeaGrid100m_cube\meta\STAC_assets
=== BMD Cube Generation Complete. Nested DataTree available at: ./bmdCube_tutorial_output/eeaGrid100m_cube ===


In [10]:
tutorial_cube = {
    "base_dir": "./bmdCube_tutorial_output/",
    "cube_name": "eeaGrid100m_cube",
    "export_as": {
        "format": "netcdf"
    },
    "spatial": {
        "target_grid": "EEA",
        "target_resolution": "100m",
        "use_bbox": True,
        "bbox": {
            "long_min": 4.326938,
            "long_max": 4.561834,
            "lat_min": 50.723880,
            "lat_max": 50.847357
        }
    },
    "temporal": {
        "start_year": 2018,
        "start_month": 1,
        "end_year": 2020,
        "end_month": 12,
        "time_aggregation": "monthly"
    },
    "sources": {
        "clms": {
            "enabled": True,
            "base_s3_path": "s3://eodata/CLMS/",
            "levels": {
                "clc": {
                    "include": True,
                    "variables": {
                        "landcover": True
                    }
                },
                "clcplus": {
                    "include": True,
                    "variables": {
                        "LULUCF-INSTANCE": True
                    }
                },
                "crop_types": {
                    "include": True,
                    "variables": {
                        "CTY": True
                    }
                },
                "dominant_leaf_type": {
                    "include": True,
                    "variables": {
                        "DLT": True
                    }
                },
                "forest_type": {
                    "include": True,
                    "variables": {
                        "FTY": True
                    }
                },
                "grassland_herbaceous": {
                    "include": True,
                    "variables": {
                        "GRA": True,
                        "HER": True
                    }
                },
                "tree_cover_density": {
                    "include": True,
                    "variables": {
                        "TCD": True
                    }
                }
            }
        }
    }
}

In [12]:
from bmc.cube.orchestrator.bmd import bmdCube
from pathlib import Path

cdse_creds = Path.home() / "cdse.json"
private_creds = Path.home() / "bmd.json"
gbif_creds = Path.home() / "gbif.json"

active_credentials = {
    "gbif_credentials": str(gbif_creds),
    "cdse_s3_credentials": str(cdse_creds),
    "private_s3_credentials": str(private_creds)
    }

orchestrator = bmdCube()
orchestrator.process_recipe(tutorial_cube, creds=active_credentials)

--- Generating Execution & Environment Provenance ---


[2026-11-17 14:08:09] [INFO] No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
INFO:No alternative execution component present in recipe. Utilizing standard execution configuration: {'min_slices_for_parallel': 16, 'max_safe_threadpool_workers': 8, 'large_slice_threshold_mb': 50.0, 'gdal_warp_block_size': 2048, 'warp_cache_cleanup_delay_sec': 30}
[2026-11-17 14:08:09] [INFO] 
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'clmscube' ===
INFO:
=== Initiating Data Cube Pipeline: 'eeaGrid100m_cube' | Dataset: 'clmscube' ===
[2026-11-17 14:08:09] [INFO]   Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
INFO:  Cube Directory   : ./bmdCube_tutorial_output/eeaGrid100m_cube
[2026-11-17 14:08:09] [INFO]   Output Directory : ./bmdCube_tutorial_output/eeaG

Initializing CLMS cube generation...


[2026-11-17 14:08:09] [INFO] Loaded catalog for level 'crop_types' (14432 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\crop_types_catalog.parquet
INFO:Loaded catalog for level 'crop_types' (14432 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\crop_types_catalog.parquet
[2026-11-17 14:08:09] [INFO] Loaded catalog for level 'dominant_leaf_type' (14432 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\dominant_leaf_type_catalog.parquet
INFO:Loaded catalog for level 'dominant_leaf_type' (14432 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\dominant_leaf_type_catalog.parquet
[2026-11-17 14:08:09] [INFO] Loaded catalog for level 'forest_type' (5412 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\forest_type_catalog.parquet
INFO:Loaded catalog for level 'forest_type' (5412 rows) from C:\Users\niels\Documents\Repositories\BmC\meta\clms\forest_type_catalog.parquet
[2026-11-17 14:08:09] [INFO] Loaded catalog for level 

Sanitizing spatial geometry...
Preparing out-of-core reprojection to EEA_100m...
Utilizing 'bilinear' resampling for reprojection.
Lazy xarray object detected. Preparing for disk stream...
Sanitizing spatial geometry...
Warping to ./bmdCube_tutorial_output/warp_cache\tree_cover_density\TCD\slice_0000.tif (Resampling: bilinear)...
Reprojection complete.
Sanitizing spatial geometry...
Preparing out-of-core reprojection to EEA_100m...
Utilizing 'bilinear' resampling for reprojection.
Lazy xarray object detected. Preparing for disk stream...
Sanitizing spatial geometry...
Warping to ./bmdCube_tutorial_output/warp_cache\tree_cover_density\TCD\slice_0001.tif (Resampling: bilinear)...


C:\Users\niels\Documents\Repositories\BmC\src\bmc\cube\spatiotemporal\raster.py:912: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  output_path = self._process_chunked_timeseries(


Reprojection complete.
Sanitizing spatial geometry...
Preparing out-of-core reprojection to EEA_100m...
Utilizing 'bilinear' resampling for reprojection.
Lazy xarray object detected. Preparing for disk stream...
Sanitizing spatial geometry...
Warping to ./bmdCube_tutorial_output/warp_cache\tree_cover_density\TCD\slice_0002.tif (Resampling: bilinear)...


C:\Users\niels\Documents\Repositories\BmC\src\bmc\cube\spatiotemporal\raster.py:912: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  output_path = self._process_chunked_timeseries(


Reprojection complete.


[2026-11-17 14:11:27] [INFO] === Assembling STAC Item for CLMSCUBE (Raster) ===
INFO:=== Assembling STAC Item for CLMSCUBE (Raster) ===
[2026-11-17 14:11:27] [INFO] Resolving target grid base for CLMS data cube generation...
INFO:Resolving target grid base for CLMS data cube generation...
[2026-11-17 14:11:27] [INFO] Resolved target grid to: EEA_100m
INFO:Resolved target grid to: EEA_100m
[2026-11-17 14:11:28] [INFO] === STAC Item Assembly Complete: eeaGrid100m_cube_clmscube_20261117141128 ===
INFO:=== STAC Item Assembly Complete: eeaGrid100m_cube_clmscube_20261117141128 ===


No STAC items available to catalog. Skipping unified STAC creation.
=== BMD Cube Generation Complete. Nested DataTree available at: ./bmdCube_tutorial_output/eeaGrid100m_cube ===


## Using the cubing engines output 

### General structure of the engine output

The general output of the cubing engine is a nested directory 

<img src="../img/datatreeDark.png" width="1000">

The engine writes all the files to a directory `base_dir/cube_name` and the general structure is 

1) A collection of folder named after the requested datasets containing the files that describe the requested variables
- For raster files this will be a set of `netcdf` or `zarr` files
- For vector data this will be a set of parquet files
2) A meta directory containing a 
- STAC directory with the STAC entry of the generated cube
- provenance json record that fingerprints the utilized software and hardware
3) A log directory containing
- log file for each dataset being processed
- a resource log file that profiles the code

In [4]:
import seedir

seedir.seedir("bmdCube_tutorial_output/eeaGrid100m_cube")

eeaGrid100m_cube/
├─chelsacube/
│ ├─annual/
│ │ ├─mymd.nc
│ │ └─qkndvi.nc
│ ├─bioclim/
│ │ ├─bio01.nc
│ │ ├─bio02.nc
│ │ ├─bio03.nc
│ │ ├─bio04.nc
│ │ ├─bio05.nc
│ │ ├─bio06.nc
│ │ ├─bio07.nc
│ │ ├─bio08.nc
│ │ ├─bio09.nc
│ │ ├─bio10.nc
│ │ ├─bio11.nc
│ │ ├─bio12.nc
│ │ ├─bio13.nc
│ │ ├─bio14.nc
│ │ ├─bio15.nc
│ │ ├─bio16.nc
│ │ ├─bio17.nc
│ │ ├─bio18.nc
│ │ └─bio19.nc
│ ├─climatologies/
│ │ ├─tas.nc
│ │ ├─tasmax.nc
│ │ └─tasmin.nc
│ └─monthly/
│   ├─tas.nc
│   ├─tasmax.nc
│   └─tasmin.nc
├─clmscube/
│ ├─clc/
│ │ └─landcover.nc
│ ├─crop_types/
│ │ └─CTY.nc
│ ├─dominant_leaf_type/
│ │ └─DLT.nc
│ ├─forest_type/
│ │ └─FTY.nc
│ └─tree_cover_density/
│   └─TCD.nc
├─gbif/
│ ├─gbif_aggregated.parquet
│ ├─gbif_EEA_100m_dimension_table.parquet
│ ├─gbif_unaggregated.parquet
│ └─validation_report/
│   └─gbif_qa_qc_failures.json
├─logs/
│ ├─eeaGrid100m_cube_chelsacube.log
│ ├─eeaGrid100m_cube_clmscube.log
│ ├─eeaGrid100m_cube_gbif.log
│ └─resource_usage.log
└─meta/
  ├─provenance_metadata.json
  └

In [6]:
from pathlib import Path

def build_file_map(base_dir_path):
    base_dir = Path(base_dir_path)
    file_map = {}
    
    # Iterate through all files recursively
    for filepath in base_dir.rglob("*"):
        if not filepath.is_file():
            continue
            
        # Target .nc, .parquet, and the specific catalog.json
        if filepath.suffix in [".nc", ".parquet"] or filepath.name == "catalog.json":
            filename = filepath.stem  # Gets filename without extension (e.g., 'tas')
            
            # Handle naming collisions (like tas.nc in different folders)
            if filename in file_map:
                # Rename the existing entry using its parent folder
                old_path = Path(file_map[filename])
                file_map[f"{old_path.parent.name}_{filename}"] = file_map.pop(filename)
                
                # Use parent folder for the new entry too
                filename = f"{filepath.parent.name}_{filename}"
                
            file_map[filename] = str(filepath.as_posix()) # as_posix() ensures forward slashes
            
    return file_map

# Run it on your directory
base_path = "bmdCube_tutorial_output/eeaGrid100m_cube"
cube_files = build_file_map(base_path)

cube_files

{'gbif_aggregated': 'bmdCube_tutorial_output/eeaGrid100m_cube/gbif/gbif_aggregated.parquet',
 'gbif_EEA_100m_dimension_table': 'bmdCube_tutorial_output/eeaGrid100m_cube/gbif/gbif_EEA_100m_dimension_table.parquet',
 'gbif_unaggregated': 'bmdCube_tutorial_output/eeaGrid100m_cube/gbif/gbif_unaggregated.parquet',
 'mymd': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/annual/mymd.nc',
 'qkndvi': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/annual/qkndvi.nc',
 'bio01': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio01.nc',
 'bio02': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio02.nc',
 'bio03': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio03.nc',
 'bio04': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio04.nc',
 'bio05': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio05.nc',
 'bio06': 'bmdCube_tutorial_output/eeaGrid100m_cube/chelsacube/bioclim/bio06.nc',
 'bio07': 'bmdCube_tutorial_output/

#### Output of rasterCube

In [15]:
import xarray as xr
import geopandas as gpd
import pandas as pd

##### Continuous variable

In [11]:
bio1_ds = xr.open_dataset(cube_files["bio01"])
bio1_ds

<xarray.Dataset> Size: 234kB
Dimensions:      (time_range: 4, scenario: 4, model: 6, y: 16, x: 19)
Coordinates:
  * time_range   (time_range) <U9 144B '1981-2010' '2011-2040' ... '2071-2100'
  * scenario     (scenario) <U10 160B 'historical' 'ssp126' 'ssp370' 'ssp585'
  * model        (model) <U13 312B 'GFDL-ESM4' 'IPSL-CM6A-LR' ... 'historical'
  * y            (y) float64 128B 3.098e+06 3.096e+06 ... 3.084e+06 3.082e+06
  * x            (x) float64 152B 3.92e+06 3.922e+06 ... 3.938e+06 3.938e+06
Data variables:
    spatial_ref  int64 8B ...
    bio01        (time_range, scenario, model, y, x) float64 233kB ...

The output of a standard continuous raster is a hypercube that has automatically stacked and aligned all relevant layers of a given variable in an indexable manner. The end result is a set of matrices that can be sliced and diced according to the end users wishes

##### Categorical variable

In [12]:
landcover_ds = xr.open_dataset(cube_files["landcover"])
landcover_ds

<xarray.Dataset> Size: 10MB
Dimensions:      (class_id: 45, time: 1, y: 151, x: 176)
Coordinates:
  * class_id     (class_id) int64 360B 1 2 3 4 5 6 7 8 ... 39 40 41 42 43 44 48
    class_name   (class_id) <U85 15kB ...
  * time         (time) datetime64[ns] 8B 2018-01-01
    provider_id  (time) <U7 28B ...
  * y            (y) float64 1kB 3.097e+06 3.097e+06 ... 3.082e+06 3.082e+06
  * x            (x) float64 1kB 3.921e+06 3.921e+06 ... 3.938e+06 3.938e+06
Data variables:
    spatial_ref  int64 8B ...
    landcover    (class_id, time, y, x) float64 10MB ...

When the cubing engine processes a categorical or discrete variable, such as the land classification in the cell above, the engine computes class fractions for each class in each cell. This is to obtain maximal information retention when switching to a lower resolution. The cube standardly contains the `class_id` and `class_name` so that users can slice the resulting hypercube more easily.

#### Output of the vectorCube

The `vectorCube` produces result that are structured in a different way to the `rasterCube` output. Because vector data is structured differently the output format significantly differs from raster data. However, raster data and vector data has been made interoperable within the cubing engine. When a user request data in a specific grid and resolution both raster data and vector data are projected to the highest shared resolution. Raster data has coordinates baked into the data structures inherently (matrix data structure described by dimensions in the meta data) whereas vector data is described tabularly. A vector cube end product can generally be described by the following standard outputs

1) Source geometries parquet file, this file provides the original data requested in its most unprocessed format
2) Aggregated/Unaggregated parquet file, this file contains the vector variables described in the shared grid to the raster data. Grid positions are denoted by a grid_idx
3) Dimension table parquet file, this file maps the grid indices back to the master grid to which all the data has been aligned and contains geometries for each cell used in the aggregated/unaggregated file

The table below lays out the relational scheme between the different outputs that are produced

<img src="../img/vectorCube_output.png" width="1000">

When reading the data the user should be mindful that aggregated data does **not** contain any geometry. If the library you use requires the parquet file to contain geometry (e.g. geopandas) be sure to utilize a parquet reading library that supports files without geometry columns

In [16]:
aggregated = pd.read_parquet(cube_files["gbif_aggregated"])
unaggregated = gpd.read_parquet(cube_files["gbif_unaggregated"])
dimension_table = gpd.read_parquet(cube_files["gbif_EEA_100m_dimension_table"])

In [17]:
aggregated

,grid_idx,speciesKey,occurrences,distinct_observers,datetime
0,963179324,6JPSJ,1,1,2018-05-01
1,963379323,75SVV,1,1,2018-05-01
2,963179324,32NH2,1,1,2018-05-01
3,963059320,53N3J,1,1,2018-05-01
4,963179324,DGP6,1,1,2018-05-01
...,...,...,...,...,...
177355,962859280,742H2,1,1,2019-11-01
177356,962899281,3G9MX,1,1,2019-11-01
177357,962779277,5Z98G,1,1,2019-11-01
177358,963099353,W8RZ,1,1,2020-11-01


The GBIF aggregated data (and all vector data) is georeferenced through a grid_idx which is linked to a specific geometry in the dimensions table

In [18]:
dimension_table

,grid_idx,geometry
0,963179324,"POLYGON ((3932500 3092000, 3932500 3092100, 39..."
1,963379323,"POLYGON ((3932400 3091500, 3932400 3091600, 39..."
2,963059320,"POLYGON ((3932100 3092300, 3932100 3092400, 39..."
3,963219324,"POLYGON ((3932500 3091900, 3932500 3092000, 39..."
4,963259334,"POLYGON ((3933500 3091800, 3933500 3091900, 39..."
...,...,...
22015,965659285,"POLYGON ((3928600 3085800, 3928600 3085900, 39..."
22016,965899274,"POLYGON ((3927500 3085200, 3927500 3085300, 39..."
22017,965779272,"POLYGON ((3927300 3085500, 3927300 3085600, 39..."
22018,965699266,"POLYGON ((3926700 3085700, 3926700 3085800, 39..."


Alternatively, the user can also consult an unaggregated file of the vector data if they enabled this setting in the recipe (`export_unaggregated` in the `aggregate` section of the vector dataset)

In [19]:
unaggregated

,gbifid,speciesKey,recordedBy,coordinateuncertaintyinmeters,geometry,passes,weight_per_point,grid_idx,src_grid_idx,datetime
0,3007561210,6JPSJ,Kristjan Adojaan,157.0,POINT (3932473.758 3092091.27),200,0.005,963179324,963179324,2018-05-01
1,3007555208,75SVV,Kristjan Adojaan,64.0,POINT (3932352.39 3091535.045),200,0.005,963379323,963379323,2018-05-01
2,3007562201,32NH2,Kristjan Adojaan,34.0,POINT (3932473.758 3092091.27),200,0.005,963179324,963179324,2018-05-01
3,3007567167,53N3J,Kristjan Adojaan,623.0,POINT (3932473.758 3092091.27),200,0.005,963059320,963179324,2018-05-01
4,3007569105,DGP6,Kristjan Adojaan,10.0,POINT (3932474.386 3092090.218),200,0.005,963179324,963179324,2018-05-01
...,...,...,...,...,...,...,...,...,...,...
198980,6497606111,742H2,Ward Tamsyn,707.0,POINT (3927568.681 3092488.944),200,0.005,962859280,963019275,2019-11-01
198981,6497607031,3G9MX,Ward Tamsyn,707.0,POINT (3927568.681 3092488.944),200,0.005,962899281,963019275,2019-11-01
198982,6497607150,5Z98G,Ward Tamsyn,707.0,POINT (3927568.681 3092488.944),200,0.005,962779277,963019275,2019-11-01
198983,6497610738,W8RZ,Frederic Gabrys,707.0,POINT (3935531.638 3091726.013),200,0.005,963099353,963299355,2020-11-01


This unaggregated dataset stores the index that the raw datapoint falls into and the new grid index that is due to the resampling that is performed during cube generation